In [1]:
%pip install -q "langchain==1.4.0" "langchain-openai==1.6.0" "langchain-text-splitters==1.1.2" "python-dotenv==1.2.3"


Note: you may need to restart the kernel to use updated packages.

<h3>개념 요약</h3>
<p>실험 하네스는 설정을 바꾸고, 색인을 다시 만들지 판단하고, 실행 결과를 같은 형식으로 기록합니다. 이 노트북은 설정 덮어쓰기, 설정 해시, 캐시 재사용, JSONL 기록을 차례로 실행합니다.</p>

In [2]:
import os

from dotenv import find_dotenv, load_dotenv

env_file = find_dotenv(usecwd=True)
assert env_file and load_dotenv(env_file), '프로젝트 루트 .env를 찾지 못했습니다.'
assert os.getenv('OPENAI_API_KEY'), 'OPENAI_API_KEY가 없습니다.'
print('환경 준비 완료')


환경 준비 완료


<h3>설정과 덮어쓰기</h3>
<p>중첩 딕셔너리에서 점으로 구분한 키 하나를 바꾸는 작은 함수를 만듭니다.</p>

In [3]:
from copy import deepcopy

config = {'split': {'chunk_size': 500, 'chunk_overlap': 50},
          'embedding': {'model': 'text-embedding-3-small'},
          'retrieval': {'k': 1}}

def override(source: dict, dotted_key: str, value: object) -> dict:
    changed = deepcopy(source)
    group, key = dotted_key.split('.')
    changed[group][key] = value
    return changed


<h3>색인 설정 해시</h3>
<p>청크와 임베딩 설정만 해시에 넣습니다. 검색 개수 <code>k</code>만 바꾸면 같은 색인을 재사용할 수 있습니다.</p>

In [4]:
import hashlib
import json


def index_key(cfg: dict) -> str:
    index_cfg = {'split': cfg['split'], 'embedding': cfg['embedding']}
    raw = json.dumps(index_cfg, sort_keys=True).encode()
    return hashlib.sha256(raw).hexdigest()[:10]

k2 = override(config, 'retrieval.k', 2)
chunk1000 = override(config, 'split.chunk_size', 1000)
print(index_key(config), index_key(k2), index_key(chunk1000))


d853036bf1 d853036bf1 8e0526f064


<h3>실제 색인과 캐시 판단</h3>
<p>작은 문서 두 건을 실제 OpenAI 임베딩으로 한 번 색인하고 해시를 키로 보관합니다. <code>k=2</code> 설정은 같은 객체를 재사용합니다.</p>

In [5]:
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

docs = [Document(page_content='휴관일은 매주 월요일입니다.', metadata={'doc_id': 'D01'}),
        Document(page_content='대출 기간은 14일입니다.', metadata={'doc_id': 'D02'})]
store = InMemoryVectorStore.from_documents(docs, OpenAIEmbeddings(model=config['embedding']['model']))
cache = {index_key(config): store}
cache_hit = index_key(k2) in cache
hits = cache[index_key(k2)].similarity_search('휴관일은 언제인가요?', k=k2['retrieval']['k'])
print('캐시 재사용:', cache_hit, '검색:', [d.metadata['doc_id'] for d in hits])


캐시 재사용: True 검색: ['D01', 'D02']


<h3>JSONL 한 행 추가하고 읽기</h3>
<p>실행 설정과 검색 결과를 JSONL 파일에 추가한 뒤 다시 읽습니다. 임시 폴더를 사용하므로 수업 폴더에 실행 로그를 남기지 않습니다.</p>

In [6]:
from pathlib import Path
from tempfile import TemporaryDirectory

temp_dir = TemporaryDirectory()
log_path = Path(temp_dir.name) / 'runs.jsonl'
record = {'name': 'k2', 'config_sha': index_key(k2),
          'cache_hit': cache_hit, 'retrieved': [d.metadata['doc_id'] for d in hits]}
with log_path.open('a', encoding='utf-8') as file:
    file.write(json.dumps(record, ensure_ascii=False) + '\n')
saved = [json.loads(line) for line in log_path.read_text(encoding='utf-8').splitlines()]
print(saved)


[{'name': 'k2', 'config_sha': 'd853036bf1', 'cache_hit': True, 'retrieved': ['D01', 'D02']}]


<h3>실행 검증</h3>
<p>설정 해시, 캐시 판단, JSONL 저장 결과를 각각 검사합니다.</p>

In [7]:
assert index_key(config) == index_key(k2)
assert index_key(config) != index_key(chunk1000)
assert cache_hit and cache[index_key(k2)] is store
assert saved == [record] and saved[0]['retrieved'] == ['D01', 'D02']
print('검증 통과: 덮어쓰기, 해시, 캐시, JSONL 기록을 확인했습니다.')


검증 통과: 덮어쓰기, 해시, 캐시, JSONL 기록을 확인했습니다.


<h3>정리</h3>
<p>검색 개수만 바꾸면 색인을 재사용하고, 청크 설정을 바꾸면 새 색인을 만듭니다. 설정 해시와 검색 결과를 같은 JSONL 행에 남기면 다음 실험의 근거를 다시 확인할 수 있습니다.</p>

<p><strong>원인</strong> — 청크 설정이 바뀌었는데도 기존 해시로 색인을 가져오면 잘못된 캐시를 재사용합니다.</p>

In [8]:
changed_config = deepcopy(config)
changed_config['split'] = {'chunk_size': 20, 'chunk_overlap': 2}
wrong_store = cache[index_key(config)]
print('잘못된 재사용:', wrong_store is store)
try:
    assert index_key(changed_config) in cache
except AssertionError:
    print('예상한 실패: 변경된 설정의 색인이 없습니다.')


잘못된 재사용: True
예상한 실패: 변경된 설정의 색인이 없습니다.


<p><strong>수정</strong> — 변경된 청크 설정으로 문서를 다시 나누고 새 색인을 만들어 해시에 저장합니다.</p>

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

settings = changed_config['split']
splitter = RecursiveCharacterTextSplitter(**settings)
rebuilt_docs = splitter.split_documents(docs)
rebuilt_store = InMemoryVectorStore.from_documents(rebuilt_docs, OpenAIEmbeddings(model=changed_config['embedding']['model']))
cache[index_key(changed_config)] = rebuilt_store
rebuilt_hits = rebuilt_store.similarity_search('휴관일', k=1)
print('After 청크:', len(rebuilt_docs), '검색:', rebuilt_hits[0].metadata['doc_id'])
assert rebuilt_store is not store
assert index_key(changed_config) in cache


After 청크: 2 검색: D01
